# 🤖 ML Package Platform (MPP)
## A No-Code Machine Learning Framework for Scientific Research

| | |
|---|---|
| **Author** | Thiago Orlando Costa Barboza |
| **Affiliation** | Ph.D. Candidate in Agronomy — UFLA / UGA |
| **ORCID** | [0000-0001-5156-2474](https://orcid.org/0000-0001-5156-2474) |
| **Lattes** | [7295109791233637](http://lattes.cnpq.br/7295109791233637) |
| **GitHub** | [thiagoagro](https://github.com/thiagoagro) |

---

**ML Package Platform (MPP)** was built to eliminate the repetitive overhead of training and comparing ML models in agricultural research — adapting scikit-learn, XGBoost, LightGBM and CatBoost across dozens of experiments for yield prediction, crop monitoring, and UAV-based phenotyping.

This notebook is a **fully self-contained, reproducible demonstration** of the complete MPP pipeline:

1. Exploratory data analysis (EDA)
2. Feature selection with **6 independent methods** + cross-method comparison
3. Model training with **16+ regression** / **15+ classification** algorithms
4. Hyperparameter optimization via **RandomizedSearchCV** and **Optuna (TPE)**
5. Performance evaluation and interactive visualization
6. One-click Excel export and model serialization

> **To convert this notebook to HTML:** run all cells, then execute `jupyter nbconvert --to html python_jupyter.ipynb`

## 📋 Table of Contents

1. [Setup & Libraries](#1-setup)
2. [Core Functions](#2-core-functions)
   - 2.1 Data Loading & EDA
   - 2.2 Evaluation Metrics
   - 2.3 Feature Scaling
   - 2.4 Machine Learning Model Catalogue
   - 2.5 Feature Selection — 6 Methods
   - 2.6 Hyperparameter Optimization (Random Search & Optuna TPE)
   - 2.7 Training Pipeline
   - 2.8 Visualization Utilities
3. [Dataset — CASP Protein Structure](#3-dataset)
4. [Feature Selection Analysis](#4-feature-selection)
5. [Model Training & HPO](#5-training)
6. [Results & Performance Evaluation](#6-results)
7. [Export & Deployment](#7-export)

---
<a id='1-setup'></a>
## ⚙️ 1. Setup & Libraries

### Required
```
pip install scikit-learn pandas numpy plotly scipy tqdm
```

### Optional (strongly recommended)
```
pip install xgboost lightgbm catboost optuna
```

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import io
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

# ── Core data science ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.stats import loguniform, uniform, randint as sp_randint

# ── scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.base import clone
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
    AdaBoostRegressor, HistGradientBoostingRegressor,
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, HistGradientBoostingClassifier
)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.svm import SVR, LinearSVR, SVC, LinearSVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge,
    HuberRegressor, LogisticRegression, RidgeClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.feature_selection import (
    mutual_info_regression, mutual_info_classif,
    f_regression, f_classif, RFE
)
from sklearn.linear_model import LassoCV

# ── Progress bar ──────────────────────────────────────────────────────────────
from tqdm.auto import tqdm

# ── Visualization ─────────────────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'   # change to 'browser' if running outside Jupyter

# ── Optional libraries ────────────────────────────────────────────────────────
_HAS_XGB = _HAS_LGBM = _HAS_CAT = _HAS_OPTUNA = False
try:
    from xgboost import XGBRegressor, XGBClassifier
    _HAS_XGB = True
except ImportError:
    pass
try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    _HAS_LGBM = True
except ImportError:
    pass
try:
    from catboost import CatBoostRegressor, CatBoostClassifier
    _HAS_CAT = True
except ImportError:
    pass
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _HAS_OPTUNA = True
except ImportError:
    pass

print(f'XGBoost  : {"✓" if _HAS_XGB    else "✗ (pip install xgboost)"}')
print(f'LightGBM : {"✓" if _HAS_LGBM   else "✗ (pip install lightgbm)"}')
print(f'CatBoost : {"✓" if _HAS_CAT    else "✗ (pip install catboost)"}')
print(f'Optuna   : {"✓" if _HAS_OPTUNA else "✗ (pip install optuna)"}')

---
<a id='2-core-functions'></a>
## 🔧 2. Core Functions

All functions below are self-contained. Run the cells sequentially — each section builds on the previous.

---
### 2.1 Data Loading & EDA

In [ ]:
def load_data(filepath) -> pd.DataFrame:
    """Load a dataset from a CSV or Excel file.

    Parameters
    ----------
    filepath : str or Path
        Path to the file. Supported formats: .csv, .xlsx, .xls

    Returns
    -------
    pd.DataFrame
        Raw dataset as a DataFrame.

    Raises
    ------
    ValueError
        If the file format is not supported.
    """
    filepath = Path(filepath)
    if filepath.suffix == '.csv':
        return pd.read_csv(filepath)
    elif filepath.suffix in ('.xlsx', '.xls'):
        return pd.read_excel(filepath)
    else:
        raise ValueError(f'Unsupported format: {filepath.suffix}. Use .csv or .xlsx.')


def generate_eda(df: pd.DataFrame) -> dict:
    """Generate a comprehensive Exploratory Data Analysis summary.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataset.

    Returns
    -------
    dict with keys:
        - 'shape'            : (rows, cols) tuple
        - 'head'             : First 5 rows
        - 'describe'         : Descriptive statistics
        - 'missing_values'   : Columns with missing value counts
        - 'numeric_cols'     : List of numeric column names
        - 'categorical_cols' : List of non-numeric column names
    """
    missing = df.isnull().sum()
    return {
        'shape'           : df.shape,
        'head'            : df.head(),
        'describe'        : df.describe(),
        'missing_values'  : missing[missing > 0],
        'numeric_cols'    : df.select_dtypes(include=[np.number]).columns.tolist(),
        'categorical_cols': df.select_dtypes(exclude=[np.number]).columns.tolist(),
    }


print('✓ Data functions defined')

---
### 2.2 Evaluation Metrics

| Metric | Task | Range | Interpretation |
|--------|------|-------|----------------|
| **R²** | Regression | 0 – 1 | Proportion of variance explained; 1 = perfect fit |
| **RMSE** | Regression | ≥ 0 | Average error in target units; lower = better |
| **MAE** | Regression | ≥ 0 | Median-robust average absolute error |
| **Willmott d** | Regression | 0 – 1 | Index of agreement; 1 = perfect, 0 = no agreement |
| **Accuracy** | Classification | 0 – 1 | Fraction of correct predictions |
| **Precision** | Classification | 0 – 1 | TP / (TP + FP) — how many predicted positives are correct |
| **Recall** | Classification | 0 – 1 | TP / (TP + FN) — how many true positives were found |
| **F1** | Classification | 0 – 1 | Harmonic mean of Precision and Recall |
| **ROC-AUC** | Classification | 0.5 – 1 | Area under the ROC curve; 1 = perfect discriminator |

> **Note on R²:** Values below 0 indicate the model performs worse than a horizontal mean line. MPP clamps these to 0 to avoid misleading negative bars in charts.

In [ ]:
def willmott_index(y_obs, y_pred) -> float:
    """Compute the Willmott Index of Agreement (d).

    The Willmott d-statistic is a dimensionless measure of model accuracy
    that is sensitive to both additive and proportional differences between
    observed and predicted values.

    Formula
    -------
    d = 1 - [ Σ(P_i - O_i)² ] / [ Σ(|P_i - Ō| + |O_i - Ō|)² ]

    Parameters
    ----------
    y_obs  : array-like — Observed (true) values.
    y_pred : array-like — Predicted values.

    Returns
    -------
    float in [0, 1]. Returns 0 if the denominator is zero.

    References
    ----------
    Willmott, C.J. (1981). On the validation of models.
    Physical Geography, 2(2), 184-194.
    """
    obs_mean = np.mean(y_obs)
    num = np.sum((y_pred - y_obs) ** 2)
    den = np.sum((np.abs(y_pred - obs_mean) + np.abs(y_obs - obs_mean)) ** 2)
    return 0.0 if den == 0 else 1.0 - (num / den)


def evaluate_regression(y_true, y_pred) -> dict:
    """Compute a full set of regression performance metrics.

    Parameters
    ----------
    y_true : array-like — Ground-truth target values.
    y_pred : array-like — Model predictions.

    Returns
    -------
    dict with keys: 'R2', 'RMSE', 'MAE', 'Willmott_d'

    Notes
    -----
    R² is clamped to [0, 1]: negative R² (model worse than the mean)
    is reported as 0 to prevent misleading visualizations.
    """
    return {
        'R2'        : max(0.0, r2_score(y_true, y_pred)),
        'RMSE'      : float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE'       : float(mean_absolute_error(y_true, y_pred)),
        'Willmott_d': float(willmott_index(y_true, y_pred)),
    }


def evaluate_classification(y_true, y_pred,
                             y_proba=None, n_classes=2,
                             average='weighted') -> dict:
    """Compute a full set of classification performance metrics.

    Parameters
    ----------
    y_true     : array-like — True class labels.
    y_pred     : array-like — Predicted class labels.
    y_proba    : array-like, optional — Predicted probabilities (for ROC-AUC).
    n_classes  : int — Number of distinct classes.
    average    : str — Averaging strategy for multi-class metrics.
                 'binary' for two-class problems, 'weighted' otherwise.

    Returns
    -------
    dict with keys: 'Accuracy', 'Precision', 'Recall', 'F1',
    and optionally 'ROC_AUC' if y_proba is provided.
    """
    metrics = {
        'Accuracy' : float(accuracy_score(y_true, y_pred)),
        'Precision': float(precision_score(y_true, y_pred, average=average, zero_division=0)),
        'Recall'   : float(recall_score(y_true, y_pred, average=average, zero_division=0)),
        'F1'       : float(f1_score(y_true, y_pred, average=average, zero_division=0)),
    }
    if y_proba is not None:
        try:
            if n_classes == 2:
                metrics['ROC_AUC'] = float(roc_auc_score(y_true, y_proba[:, 1]))
            else:
                metrics['ROC_AUC'] = float(roc_auc_score(y_true, y_proba,
                                                           multi_class='ovr'))
        except Exception:
            pass
    return metrics


print('✓ Evaluation functions defined')

---
### 2.3 Feature Scaling

Scaling transforms all features to a common range, preventing models sensitive to magnitude (SVM, KNN, MLP, linear models) from being dominated by high-variance columns.

| Method | Formula | Best for |
|--------|---------|----------|
| **StandardScaler** | (x − μ) / σ | Normally distributed data; most general choice |
| **MinMaxScaler** | (x − min) / (max − min) | When an explicit [0, 1] range is required |
| **RobustScaler** | (x − median) / IQR | Data with significant outliers |
| **None** | — | Tree-based models (RF, XGBoost) that are scale-invariant |

In [ ]:
def scale_features(X: pd.DataFrame, method: str = 'StandardScaler'):
    """Scale input features using the specified normalisation strategy.

    Parameters
    ----------
    X      : pd.DataFrame — Numeric feature matrix.
    method : str — One of 'StandardScaler', 'MinMaxScaler',
                   'RobustScaler', or 'None'.

    Returns
    -------
    X_scaled : pd.DataFrame — Transformed features with original column names.
    scaler   : fitted scaler object, or None if method is 'None'.

    Raises
    ------
    ValueError if method is not one of the supported options.
    """
    scalers = {
        'StandardScaler': StandardScaler(),
        'MinMaxScaler'  : MinMaxScaler(),
        'RobustScaler'  : RobustScaler(),
    }
    if method in ('None', None):
        return X, None
    if method not in scalers:
        raise ValueError(f'Unsupported scaling method: {method!r}. '
                         f'Choose from {list(scalers.keys())} or "None".')
    scaler = scalers[method]
    X_scaled = pd.DataFrame(scaler.fit_transform(X),
                             columns=X.columns, index=X.index)
    return X_scaled, scaler


print('✓ Scaling function defined')

---
### 2.4 Machine Learning Model Catalogue

MPP includes **16 regression** and **15 classification** algorithms out of the box.

#### Regression models
| Category | Models |
|----------|--------|
| Tree ensembles | Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting, AdaBoost, Decision Tree |
| Boosting (optional) | XGBoost, LightGBM, CatBoost |
| Linear / Regularised | Multilinear, Ridge, Lasso, ElasticNet, Bayesian Ridge, Huber |
| Other | SVM (RBF), LinearSVR, KNN, MLP |

#### Classification models
| Category | Models |
|----------|--------|
| Tree ensembles | Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting, AdaBoost, Decision Tree |
| Boosting (optional) | XGBoost, LightGBM, CatBoost |
| Linear / Probabilistic | Logistic Regression, Ridge Classifier, Gaussian NB, LDA, QDA |
| Other | SVM (RBF), LinearSVC, KNN, MLP |

In [ ]:
def get_regression_models() -> dict:
    """Return a dictionary of unfitted regression estimators.

    Each model is initialised with sensible defaults suitable for a
    first-pass comparison. Hyperparameters can be further optimised via
    RandomizedSearchCV or Optuna (see Section 2.6).

    Returns
    -------
    dict mapping model_name (str) -> unfitted sklearn-compatible estimator.
    XGBoost, LightGBM, and CatBoost are included automatically if installed.
    """
    mdls = {
        'Random Forest'       : RandomForestRegressor(n_estimators=300, max_features='sqrt',
                                                       random_state=42, n_jobs=-1),
        'Extra Trees'         : ExtraTreesRegressor(n_estimators=300, max_features='sqrt',
                                                     bootstrap=False, random_state=42, n_jobs=-1),
        'Gradient Boosting'   : GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                                           max_depth=3, random_state=42),
        'HistGradientBoosting': HistGradientBoostingRegressor(learning_rate=0.05, max_iter=500,
                                                               random_state=42),
        'AdaBoost'            : AdaBoostRegressor(n_estimators=300, learning_rate=0.05,
                                                   random_state=42),
        'Decision Tree'       : DecisionTreeRegressor(random_state=42),
        'SVM (RBF)'           : SVR(kernel='rbf', C=10.0, epsilon=0.1, gamma='scale'),
        'LinearSVR'           : LinearSVR(C=1.0, epsilon=0.1, max_iter=5000, random_state=42),
        'KNN'                 : KNeighborsRegressor(n_neighbors=5, weights='distance', n_jobs=-1),
        'MLP'                 : MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=2000,
                                             random_state=42),
        'Multlinear regression': LinearRegression(),
        'Ridge'               : Ridge(alpha=1.0),
        'Lasso'               : Lasso(alpha=1e-3, max_iter=5000),
        'ElasticNet'          : ElasticNet(alpha=1e-3, max_iter=5000),
        'Bayesian Ridge'      : BayesianRidge(max_iter=300),
        'Huber Regressor'     : HuberRegressor(max_iter=5000),
    }
    if _HAS_XGB:
        mdls['XGBoost']  = XGBRegressor(n_estimators=300, learning_rate=0.05,
                                         max_depth=6, random_state=42,
                                         n_jobs=-1, verbosity=0)
    if _HAS_LGBM:
        mdls['LightGBM'] = LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                          num_leaves=31, random_state=42,
                                          n_jobs=-1, verbose=-1)
    if _HAS_CAT:
        mdls['CatBoost'] = CatBoostRegressor(iterations=500, learning_rate=0.05,
                                              depth=6, random_state=42, verbose=0)
    return mdls


def get_classification_models() -> dict:
    """Return a dictionary of unfitted classification estimators.

    Returns
    -------
    dict mapping model_name (str) -> unfitted sklearn-compatible estimator.
    XGBoost, LightGBM, and CatBoost are included automatically if installed.
    """
    mdls = {
        'Random Forest'       : RandomForestClassifier(n_estimators=300, random_state=42,
                                                        n_jobs=-1),
        'Extra Trees'         : ExtraTreesClassifier(n_estimators=300, random_state=42,
                                                      n_jobs=-1),
        'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=500, learning_rate=0.05,
                                                            random_state=42),
        'HistGradientBoosting': HistGradientBoostingClassifier(learning_rate=0.05, max_iter=500,
                                                                random_state=42),
        'AdaBoost'            : AdaBoostClassifier(n_estimators=300, learning_rate=0.05,
                                                    random_state=42),
        'Decision Tree'       : DecisionTreeClassifier(random_state=42),
        'SVM (RBF)'           : SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
        'LinearSVC'           : LinearSVC(C=1.0, max_iter=5000, random_state=42),
        'KNN'                 : KNeighborsClassifier(n_neighbors=5, weights='distance',
                                                     n_jobs=-1),
        'MLP'                 : MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=2000,
                                              random_state=42),
        'Logistic Regression' : LogisticRegression(max_iter=5000, random_state=42),
        'Ridge Classifier'    : RidgeClassifier(),
        'Gaussian NB'         : GaussianNB(),
        'LDA'                 : LinearDiscriminantAnalysis(),
        'QDA'                 : QuadraticDiscriminantAnalysis(),
    }
    if _HAS_XGB:
        mdls['XGBoost']  = XGBClassifier(n_estimators=300, learning_rate=0.05,
                                          random_state=42, n_jobs=-1,
                                          eval_metric='logloss', verbosity=0)
    if _HAS_LGBM:
        mdls['LightGBM'] = LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                           random_state=42, n_jobs=-1, verbose=-1)
    if _HAS_CAT:
        mdls['CatBoost'] = CatBoostClassifier(iterations=500, learning_rate=0.05,
                                               random_state=42, verbose=0)
    return mdls


print(f'Regression models  : {len(get_regression_models())}')
print(f'Classification models: {len(get_classification_models())}')

---
### 2.5 Feature Selection — 6 Methods

Feature selection reduces dimensionality, removes noise, and improves model generalisation. MPP implements **6 independent methods** from scikit-learn, each with a different statistical foundation:

| Method | Approach | Best for |
|--------|----------|----------|
| **ExtraTrees** | Tree-based impurity reduction | Non-linear, high-dimensional data |
| **Random Forest** | Tree-based impurity reduction | Robust baseline for tabular data |
| **Mutual Information** | Non-parametric dependency measure | Non-linear, mixed feature types |
| **F-test** | ANOVA / F-regression | Linear relationships |
| **RFE** | Recursive Feature Elimination | When feature count matters |
| **Lasso / L1** | Regularisation-based shrinkage | Sparse, linearly correlated features |

All methods return a **normalised score (0–1)** for direct cross-method comparison.

In [ ]:
# Human-readable label → internal key
FEATURE_SELECTION_METHODS = {
    'ExtraTrees Importance'          : 'extratrees',
    'Random Forest Importance'       : 'random_forest',
    'Mutual Information'             : 'mutual_info',
    'F-test (ANOVA / F-regression)'  : 'ftest',
    'RFE — Random Forest'            : 'rfe',
    'Lasso / L1 Regularization'      : 'lasso',
}


def _prep(X: pd.DataFrame, y: pd.Series):
    """Fill NaN values and align X/y indices before feature selection."""
    X_f = X.fillna(X.mean())
    idx = X_f.index.intersection(y.dropna().index)
    return X_f.loc[idx], y.loc[idx]


def _to_df(features, scores, method_name: str) -> pd.DataFrame:
    """Convert raw importance scores to a standardised output DataFrame.

    Normalises scores to [0, 1], adds rank and cumulative importance.

    Returns
    -------
    pd.DataFrame with columns:
        Feature, Score, Normalized_Score, Rank, Method, Cumulative_Importance
    """
    df = pd.DataFrame({'Feature': list(features), 'Score': list(scores)})
    df = df.sort_values('Score', ascending=False).reset_index(drop=True)
    lo, hi = df['Score'].min(), df['Score'].max()
    df['Normalized_Score'] = (df['Score'] - lo) / (hi - lo) if hi > lo else 1.0
    df['Rank'] = range(1, len(df) + 1)
    df['Method'] = method_name
    total = df['Normalized_Score'].sum()
    df['Cumulative_Importance'] = df['Normalized_Score'].cumsum() / total if total > 0 else 0.0
    return df


def _fs_extratrees(X, y, task_type):
    """ExtraTrees feature importance (tree-based, non-parametric)."""
    Cls = ExtraTreesRegressor if task_type == 'Regression' else ExtraTreesClassifier
    X_f, y_f = _prep(X, y)
    m = Cls(n_estimators=100, random_state=42, n_jobs=-1)
    m.fit(X_f, y_f)
    return _to_df(X.columns, m.feature_importances_, 'ExtraTrees')


def _fs_random_forest(X, y, task_type):
    """Random Forest feature importance (tree-based, bootstrap-aggregated)."""
    Cls = RandomForestRegressor if task_type == 'Regression' else RandomForestClassifier
    X_f, y_f = _prep(X, y)
    m = Cls(n_estimators=100, random_state=42, n_jobs=-1)
    m.fit(X_f, y_f)
    return _to_df(X.columns, m.feature_importances_, 'Random Forest')


def _fs_mutual_info(X, y, task_type):
    """Mutual Information — measures non-linear statistical dependency."""
    X_f, y_f = _prep(X, y)
    fn = mutual_info_regression if task_type == 'Regression' else mutual_info_classif
    scores = fn(X_f, y_f, random_state=42)
    return _to_df(X.columns, scores, 'Mutual Information')


def _fs_ftest(X, y, task_type):
    """F-test (ANOVA for classification, F-regression for regression).

    Measures the strength of linear relationship between each feature
    and the target. Best for linear problems.
    """
    X_f, y_f = _prep(X, y)
    fn = f_regression if task_type == 'Regression' else f_classif
    f_scores, _ = fn(X_f, y_f)
    f_scores = np.nan_to_num(f_scores, nan=0.0)
    return _to_df(X.columns, f_scores, 'F-test')


def _fs_rfe(X, y, task_type):
    """Recursive Feature Elimination (RFE) using Random Forest.

    Iteratively removes the least important feature until half remain.
    Rank 1 = most important.
    """
    X_f, y_f = _prep(X, y)
    Cls = RandomForestRegressor if task_type == 'Regression' else RandomForestClassifier
    estimator = Cls(n_estimators=50, random_state=42, n_jobs=-1)
    selector = RFE(estimator, n_features_to_select=max(1, X_f.shape[1] // 2), step=1)
    selector.fit(X_f, y_f)
    # Invert ranking so higher score = more important
    scores = (selector.ranking_.max() + 1) - selector.ranking_
    return _to_df(X.columns, scores.astype(float), 'RFE')


def _fs_lasso(X, y, task_type):
    """Lasso / L1 regularisation — drives irrelevant feature coefficients to zero.

    Uses LassoCV (cross-validated alpha) for regression and
    L1-penalised LogisticRegression for classification.
    """
    from sklearn.preprocessing import StandardScaler as _SS
    X_f, y_f = _prep(X, y)
    Xs = _SS().fit_transform(X_f)
    if task_type == 'Regression':
        m = LassoCV(cv=5, random_state=42, max_iter=5000)
        m.fit(Xs, y_f)
        scores = np.abs(m.coef_)
    else:
        m = LogisticRegression(penalty='l1', solver='saga', C=0.1,
                               max_iter=2000, random_state=42)
        m.fit(Xs, y_f)
        scores = np.abs(m.coef_).mean(axis=0)
    return _to_df(X.columns, scores, 'Lasso / L1')


_FS_FNS = {
    'extratrees'  : _fs_extratrees,
    'random_forest': _fs_random_forest,
    'mutual_info' : _fs_mutual_info,
    'ftest'       : _fs_ftest,
    'rfe'         : _fs_rfe,
    'lasso'       : _fs_lasso,
}


def run_feature_selection(X: pd.DataFrame, y: pd.Series,
                          task_type: str, method_key: str) -> pd.DataFrame:
    """Run a single feature selection method and return a ranked DataFrame.

    Parameters
    ----------
    X          : pd.DataFrame — Numeric feature matrix (no target column).
    y          : pd.Series   — Target variable.
    task_type  : str         — 'Regression' or 'Classification'.
    method_key : str         — Key from FEATURE_SELECTION_METHODS values.

    Returns
    -------
    pd.DataFrame with columns:
        Feature, Score, Normalized_Score (0–1), Rank, Method, Cumulative_Importance

    Example
    -------
    >>> result = run_feature_selection(X, y, 'Regression', 'extratrees')
    >>> print(result.head())
    """
    fn = _FS_FNS.get(method_key)
    if fn is None:
        raise ValueError(f'Unknown method key: {method_key!r}. '
                         f'Valid keys: {list(_FS_FNS.keys())}')
    return fn(X, y, task_type)


print('✓ Feature selection functions defined (6 methods)')

---
### 2.6 Hyperparameter Optimization (HPO)

Hyperparameters are settings **not learned from data** — they control the learning process itself (e.g., `n_estimators`, `learning_rate`, `C`). Choosing them well is one of the biggest levers for improving model performance.

#### Available strategies

| Strategy | How it works | Best for |
|----------|-------------|----------|
| **Disabled** | Uses default parameters | Quick baseline comparison |
| **Random Search** | Randomly samples from param distributions (`n_iter=25, cv=5`) | Good baseline HPO; fast |
| **Optuna (TPE)** | Bayesian search via Tree-structured Parzen Estimators — learns from past trials to focus on promising regions | Best results; especially effective for 20+ trials |

#### Why Optuna TPE beats Random Search

Random Search samples blindly from the entire parameter space. TPE fits two probability models — `l(x)` (good param regions) and `g(x)` (bad regions) — and proposes the next trial by maximising the Expected Improvement ratio `l(x)/g(x)`. After ~10 trials it concentrates sampling on the best-performing regions.

> **Runtime tip:** For the notebook demo, `n_trials=15` is used. For production, `n_trials=50–100` per model is recommended.

In [ ]:
def _get_param_grid(model_name: str, task_type: str = 'Regression') -> dict:
    """Return a parameter distribution dict for RandomizedSearchCV.

    Uses scipy continuous distributions (loguniform, uniform) for
    proper log-scale and linear sampling — significantly better than
    discrete lists for continuous hyperparameters.

    Parameters
    ----------
    model_name : str — Exact model name from get_regression/classification_models().
    task_type  : str — 'Regression' or 'Classification'.

    Returns
    -------
    dict mapping param_name -> distribution or list.
    Returns {} for models with no meaningful hyperparameters.
    """
    if model_name == 'Random Forest':
        return {'n_estimators': sp_randint(100, 1000),
                'max_depth'   : [None, 5, 10, 15, 20, 30],
                'min_samples_split': sp_randint(2, 20),
                'min_samples_leaf' : sp_randint(1, 10),
                'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
                'bootstrap'   : [True, False]}
    if model_name == 'Extra Trees':
        return {'n_estimators': sp_randint(100, 1000),
                'max_depth'   : [None, 5, 10, 15, 20, 30],
                'min_samples_split': sp_randint(2, 20),
                'min_samples_leaf' : sp_randint(1, 10),
                'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7]}
    if model_name == 'Gradient Boosting':
        return {'learning_rate': loguniform(0.005, 0.3),
                'n_estimators' : sp_randint(100, 600),
                'max_depth'    : sp_randint(2, 9),
                'subsample'    : uniform(0.6, 0.4),
                'min_samples_split': sp_randint(2, 20),
                'max_features' : ['sqrt', 'log2', None]}
    if model_name == 'HistGradientBoosting':
        return {'learning_rate'    : loguniform(0.005, 0.3),
                'max_iter'        : sp_randint(100, 1000),
                'max_depth'       : [None, 3, 5, 7, 10, 15],
                'min_samples_leaf': sp_randint(10, 100),
                'l2_regularization': loguniform(1e-4, 10.0),
                'max_bins'        : [63, 127, 255]}
    if model_name == 'AdaBoost':
        return {'n_estimators' : sp_randint(50, 500),
                'learning_rate': loguniform(0.01, 2.0)}
    if model_name == 'Decision Tree':
        return {'max_depth'   : [None, 3, 5, 7, 10, 15, 20],
                'min_samples_split': sp_randint(2, 30),
                'min_samples_leaf' : sp_randint(1, 15),
                'max_features': ['sqrt', 'log2', None]}
    if model_name == 'XGBoost':
        return {'learning_rate'   : loguniform(0.005, 0.3),
                'n_estimators'    : sp_randint(100, 1000),
                'max_depth'       : sp_randint(3, 11),
                'subsample'       : uniform(0.5, 0.5),
                'colsample_bytree': uniform(0.5, 0.5),
                'reg_alpha'       : loguniform(1e-3, 10.0),
                'reg_lambda'      : loguniform(0.1, 10.0),
                'min_child_weight': sp_randint(1, 10),
                'gamma'           : uniform(0.0, 5.0)}
    if model_name == 'LightGBM':
        return {'learning_rate'   : loguniform(0.005, 0.3),
                'n_estimators'    : sp_randint(100, 1000),
                'num_leaves'      : sp_randint(20, 200),
                'min_child_samples': sp_randint(5, 100),
                'feature_fraction': uniform(0.5, 0.5),
                'bagging_fraction': uniform(0.5, 0.5),
                'bagging_freq'    : sp_randint(1, 8),
                'reg_alpha'       : loguniform(1e-3, 10.0),
                'reg_lambda'      : loguniform(1e-3, 10.0)}
    if model_name == 'CatBoost':
        return {'learning_rate'     : loguniform(0.005, 0.3),
                'iterations'        : sp_randint(100, 1000),
                'depth'             : sp_randint(3, 11),
                'l2_leaf_reg'       : sp_randint(1, 10),
                'bagging_temperature': uniform(0.0, 1.0),
                'random_strength'   : uniform(0.0, 10.0)}
    if model_name == 'SVM (RBF)':
        grid = {'C': loguniform(0.01, 1000.0), 'gamma': ['scale', 'auto']}
        if task_type == 'Regression':
            grid['epsilon'] = loguniform(1e-3, 1.0)
        return grid
    if model_name in ('LinearSVR', 'LinearSVC'):
        return {'C': loguniform(0.01, 100.0)}
    if model_name == 'KNN':
        return {'n_neighbors': sp_randint(3, 30),
                'weights'    : ['uniform', 'distance'],
                'metric'     : ['euclidean', 'manhattan', 'minkowski']}
    if model_name == 'MLP':
        return {'hidden_layer_sizes': [(64,), (128,), (256,), (64, 64),
                                        (128, 64), (128, 128), (256, 128, 64)],
                'activation'        : ['relu', 'tanh'],
                'alpha'             : loguniform(1e-5, 0.1),
                'learning_rate_init': loguniform(1e-4, 0.01)}
    if model_name in ('Ridge', 'Bayesian Ridge'):
        return {'alpha': loguniform(1e-3, 1000.0)}
    if model_name == 'Lasso':
        return {'alpha': loguniform(1e-5, 10.0)}
    if model_name == 'ElasticNet':
        return {'alpha': loguniform(1e-5, 10.0), 'l1_ratio': uniform(0.05, 0.90)}
    if model_name == 'Huber Regressor':
        return {'epsilon': uniform(1.05, 3.0), 'alpha': loguniform(1e-5, 10.0)}
    if model_name == 'Logistic Regression':
        return {'C': loguniform(1e-3, 100.0)}
    if model_name == 'Ridge Classifier':
        return {'alpha': loguniform(1e-3, 1000.0)}
    return {}   # LinearRegression, GaussianNB, LDA, QDA — no meaningful HPO


def _run_random_search(model, param_grid: dict, X_train, y_train):
    """Fit RandomizedSearchCV and return the best estimator.

    Uses 25 iterations with 5-fold CV for a good exploration/cost trade-off.
    scipy distributions in param_grid enable continuous log-uniform sampling.
    """
    search = RandomizedSearchCV(model, param_grid, n_iter=25, cv=5,
                                 random_state=42, n_jobs=-1)
    search.fit(X_train, y_train)
    return search.best_estimator_


def _run_optuna(model, model_name: str, X_train, y_train,
                task_type: str, n_trials: int):
    """Run Optuna TPE search and return (best_fitted_model, study).

    Tree-structured Parzen Estimators (TPE) build probabilistic models of
    the objective and use Expected Improvement to propose the next trial.
    Consistently outperforms random search after ~15–20 trials.

    Parameters
    ----------
    model      : unfitted estimator
    model_name : str — used to select the parameter space
    X_train    : training features
    y_train    : training targets
    task_type  : 'Regression' or 'Classification'
    n_trials   : number of Optuna trials (30–100 recommended for production)

    Returns
    -------
    best_model : fitted estimator with best found hyperparameters
    study      : optuna.Study object (contains all trial history)
    """
    if not _HAS_OPTUNA:
        raise ImportError('Optuna not installed. Run: pip install optuna')

    def objective(trial):
        if model_name == 'Random Forest':
            params = {'n_estimators'    : trial.suggest_int('n_estimators', 100, 1000),
                      'max_depth'       : trial.suggest_categorical('max_depth',
                                             [None, 5, 10, 15, 20, 30]),
                      'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                      'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 10),
                      'max_features'    : trial.suggest_categorical('max_features',
                                             ['sqrt', 'log2', 0.3, 0.5, 0.7]),
                      'bootstrap'       : trial.suggest_categorical('bootstrap',
                                             [True, False])}
        elif model_name == 'Extra Trees':
            params = {'n_estimators'    : trial.suggest_int('n_estimators', 100, 1000),
                      'max_depth'       : trial.suggest_categorical('max_depth',
                                             [None, 5, 10, 15, 20, 30]),
                      'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                      'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 10),
                      'max_features'    : trial.suggest_categorical('max_features',
                                             ['sqrt', 'log2', 0.3, 0.5, 0.7])}
        elif model_name == 'Gradient Boosting':
            params = {'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                      'n_estimators' : trial.suggest_int('n_estimators', 100, 600),
                      'max_depth'    : trial.suggest_int('max_depth', 2, 8),
                      'subsample'    : trial.suggest_float('subsample', 0.6, 1.0),
                      'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                      'max_features' : trial.suggest_categorical('max_features',
                                          ['sqrt', 'log2', None])}
        elif model_name == 'XGBoost':
            params = {'learning_rate'   : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                      'n_estimators'    : trial.suggest_int('n_estimators', 100, 1000),
                      'max_depth'       : trial.suggest_int('max_depth', 3, 10),
                      'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
                      'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                      'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
                      'reg_lambda'      : trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
                      'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                      'gamma'           : trial.suggest_float('gamma', 0.0, 5.0)}
        elif model_name == 'LightGBM':
            params = {'learning_rate'   : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                      'n_estimators'    : trial.suggest_int('n_estimators', 100, 1000),
                      'num_leaves'      : trial.suggest_int('num_leaves', 20, 200),
                      'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                      'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
                      'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
                      'bagging_freq'    : trial.suggest_int('bagging_freq', 1, 7),
                      'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
                      'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)}
        elif model_name == 'CatBoost':
            params = {'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                      'iterations'        : trial.suggest_int('iterations', 100, 1000),
                      'depth'             : trial.suggest_int('depth', 3, 10),
                      'l2_leaf_reg'       : trial.suggest_int('l2_leaf_reg', 1, 10),
                      'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
                      'random_strength'   : trial.suggest_float('random_strength', 0.0, 10.0)}
        elif model_name == 'SVM (RBF)':
            params = {'C'    : trial.suggest_float('C', 0.01, 1000.0, log=True),
                      'gamma': trial.suggest_categorical('gamma', ['scale', 'auto'])}
            if task_type == 'Regression':
                params['epsilon'] = trial.suggest_float('epsilon', 1e-3, 1.0, log=True)
        elif model_name == 'KNN':
            params = {'n_neighbors': trial.suggest_int('n_neighbors', 3, 30),
                      'weights'    : trial.suggest_categorical('weights', ['uniform', 'distance']),
                      'metric'     : trial.suggest_categorical('metric',
                                        ['euclidean', 'manhattan', 'minkowski'])}
        elif model_name == 'MLP':
            params = {'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes',
                                                [(64,), (128,), (64, 64), (128, 64),
                                                 (128, 128), (256, 128, 64)]),
                      'activation'        : trial.suggest_categorical('activation', ['relu', 'tanh']),
                      'alpha'             : trial.suggest_float('alpha', 1e-5, 0.1, log=True),
                      'learning_rate_init': trial.suggest_float('learning_rate_init',
                                                1e-4, 0.01, log=True)}
        elif model_name in ('Ridge', 'Bayesian Ridge'):
            params = {'alpha': trial.suggest_float('alpha', 1e-3, 1000.0, log=True)}
        elif model_name == 'Lasso':
            params = {'alpha': trial.suggest_float('alpha', 1e-5, 10.0, log=True)}
        elif model_name == 'ElasticNet':
            params = {'alpha'   : trial.suggest_float('alpha', 1e-5, 10.0, log=True),
                      'l1_ratio': trial.suggest_float('l1_ratio', 0.05, 0.95)}
        elif model_name == 'Logistic Regression':
            params = {'C': trial.suggest_float('C', 1e-3, 100.0, log=True)}
        else:
            return 0.0   # no HPO for LinearRegression, GaussianNB, LDA, QDA

        m = clone(model)
        m.set_params(**params)
        scoring = 'r2' if task_type == 'Regression' else 'accuracy'
        scores = cross_val_score(m, X_train, y_train, cv=3,
                                  scoring=scoring, n_jobs=-1)
        return float(scores.mean())

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_model = clone(model)
    if study.best_params:
        best_model.set_params(**study.best_params)
    best_model.fit(X_train, y_train)
    return best_model, study


print('✓ HPO functions defined (RandomizedSearchCV + Optuna TPE)')

---
### 2.7 Training Pipeline

The pipeline orchestrates the full workflow in a single function call:

```
raw data → NaN removal → scaling → train/test split
         → model loop → [optional HPO] → fit → evaluate
         → results DataFrame + predictions DataFrame
```

In [ ]:
def run_training_pipeline(
    df: pd.DataFrame,
    features_cols: list,
    target_col: str,
    task_type: str = 'Regression',
    scaler_option: str = 'StandardScaler',
    test_size: float = 0.3,
    selected_models: list = None,
    hpo_method: str = 'none',
    n_trials: int = 30,
):
    """Run the complete ML training pipeline end-to-end.

    Parameters
    ----------
    df              : pd.DataFrame — Full dataset (features + target).
    features_cols   : list[str]   — Column names to use as input features.
    target_col      : str         — Name of the target column.
    task_type       : str         — 'Regression' or 'Classification'.
    scaler_option   : str         — 'StandardScaler' | 'MinMaxScaler' |
                                    'RobustScaler' | 'None'.
    test_size       : float       — Fraction of data for the test set (default 0.3).
    selected_models : list[str]   — Model names to train. None = all available.
    hpo_method      : str         — 'none' | 'random_search' | 'optuna'.
    n_trials        : int         — Optuna trials per model (ignored otherwise).

    Returns
    -------
    results_df     : pd.DataFrame — Per-model Train/Test metrics, sorted by
                                    Test_R2 (regression) or Test_Accuracy (classification).
    predictions_df : pd.DataFrame — Observed vs Predicted for every model and split.
    trained_models : dict         — model_name -> fitted estimator.
    optuna_studies : dict or None — model_name -> optuna.Study (only if hpo_method='optuna').

    Example
    -------
    >>> results, preds, models, studies = run_training_pipeline(
    ...     df=df, features_cols=['F1','F4','F6'], target_col='RMSD',
    ...     task_type='Regression', hpo_method='optuna', n_trials=20,
    ...     selected_models=['Random Forest', 'XGBoost']
    ... )
    >>> print(results[['Model', 'Test_R2', 'Test_RMSE']])
    """
    # ── 1. Clean data ─────────────────────────────────────────────────────────
    X = df[features_cols]
    y = df[target_col]
    valid_idx = X.dropna().index.intersection(y.dropna().index)
    X, y = X.loc[valid_idx], y.loc[valid_idx]

    # ── 2. Scale ──────────────────────────────────────────────────────────────
    X_scaled, _ = scale_features(X, method=scaler_option)

    # ── 3. Split ──────────────────────────────────────────────────────────────
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42
    )

    # ── 4. Model dictionary ───────────────────────────────────────────────────
    if task_type == 'Regression':
        model_dict = get_regression_models()
    else:
        model_dict = get_classification_models()
        n_classes  = len(np.unique(y))
        avg_method = 'binary' if n_classes == 2 else 'weighted'

    if selected_models:
        model_dict = {k: v for k, v in model_dict.items() if k in selected_models}

    results_list   = []
    predictions_list = []
    trained_models = {}
    optuna_studies = {} if hpo_method == 'optuna' else None

    # ── 5. Training loop ──────────────────────────────────────────────────────
    for name, model in tqdm(model_dict.items(), desc='Training', unit='model'):
        try:
            # ── HPO ───────────────────────────────────────────────────────────
            if hpo_method == 'random_search':
                param_grid = _get_param_grid(name, task_type)
                model = (_run_random_search(model, param_grid, X_train, y_train)
                         if param_grid else model.fit(X_train, y_train) or model)
            elif hpo_method == 'optuna':
                model, study = _run_optuna(model, name, X_train, y_train,
                                            task_type, n_trials)
                if study.best_params:
                    optuna_studies[name] = study
            else:
                model.fit(X_train, y_train)

            trained_models[name] = model

            # ── Predict ───────────────────────────────────────────────────────
            y_tr_pred = model.predict(X_train)
            y_te_pred = model.predict(X_test)
            y_tr_proba = y_te_proba = None
            if task_type == 'Classification' and hasattr(model, 'predict_proba'):
                y_tr_proba = model.predict_proba(X_train)
                y_te_proba = model.predict_proba(X_test)

            # ── Metrics ───────────────────────────────────────────────────────
            if task_type == 'Regression':
                tr = evaluate_regression(y_train, y_tr_pred)
                te = evaluate_regression(y_test,  y_te_pred)
            else:
                tr = evaluate_classification(y_train, y_tr_pred,
                                              y_tr_proba, n_classes, avg_method)
                te = evaluate_classification(y_test,  y_te_pred,
                                              y_te_proba, n_classes, avg_method)

            row = {'Model': name}
            row.update({f'Train_{k}': v for k, v in tr.items()})
            row.update({f'Test_{k}' : v for k, v in te.items()})
            results_list.append(row)

            # ── Predictions ───────────────────────────────────────────────────
            predictions_list.append(pd.concat([
                pd.DataFrame({'Model': name, 'Dataset': 'Train',
                               'Observed': np.array(y_train),
                               'Predicted': np.array(y_tr_pred)}),
                pd.DataFrame({'Model': name, 'Dataset': 'Test',
                               'Observed': np.array(y_test),
                               'Predicted': np.array(y_te_pred)}),
            ]))
        except Exception as exc:
            print(f'  [WARNING] {name}: {exc}')

    results_df = pd.DataFrame(results_list)
    sort_col   = 'Test_R2' if task_type == 'Regression' else 'Test_Accuracy'
    if sort_col in results_df.columns:
        results_df = results_df.sort_values(sort_col, ascending=False)

    predictions_df = pd.concat(predictions_list, ignore_index=True)
    return results_df, predictions_df, trained_models, optuna_studies


print('✓ Training pipeline defined')

---
### 2.8 Visualization Utilities

All charts are built with **Plotly** for interactive HTML output. White backgrounds ensure readability when embedded in publications or websites.

In [ ]:
_WHITE = dict(paper_bgcolor='white', plot_bgcolor='white',
              font=dict(color='#111111', size=14))
_AXIS  = dict(gridcolor='#e0e0e0', linecolor='#999999',
              tickfont=dict(color='#111111', size=13),
              title_font=dict(color='#111111', size=14))


def _apply_white(fig):
    """Apply white background and black axis labels to a Plotly figure."""
    fig.update_layout(**_WHITE)
    fig.update_xaxes(**_AXIS)
    fig.update_yaxes(**_AXIS)
    return fig


def plot_correlation_matrix(df: pd.DataFrame):
    """Return a Plotly heatmap of the Pearson correlation matrix.

    Parameters
    ----------
    df : pd.DataFrame — Dataset (non-numeric columns are dropped automatically).
    """
    corr = df.select_dtypes(include=[np.number]).corr()
    fig  = px.imshow(corr, text_auto='.2f', aspect='auto',
                     color_continuous_scale='RdBu_r',
                     title='Correlation Matrix')
    fig.update_layout(height=600, **_WHITE)
    return fig


def plot_scatter(df: pd.DataFrame, x_col: str, y_col: str):
    """Return an interactive scatter plot with OLS trendline.

    Parameters
    ----------
    df    : pd.DataFrame
    x_col : str — Column for X axis.
    y_col : str — Column for Y axis.
    """
    fig = px.scatter(df, x=x_col, y=y_col,
                     title=f'Scatter: {x_col} vs {y_col}',
                     trendline='ols', opacity=0.6)
    return _apply_white(fig)


def plot_histogram(df: pd.DataFrame, col: str):
    """Return an interactive histogram with marginal box plot.

    Parameters
    ----------
    df  : pd.DataFrame
    col : str — Column to plot.
    """
    fig = px.histogram(df, x=col, marginal='box',
                       title=f'Distribution: {col}')
    return _apply_white(fig)


def plot_fs_single(imp_df: pd.DataFrame, method_name: str):
    """Horizontal bar chart for one feature selection method.

    Parameters
    ----------
    imp_df      : pd.DataFrame — Output of run_feature_selection().
    method_name : str          — Label for the chart title.
    """
    df  = imp_df.sort_values('Normalized_Score', ascending=True)
    fig = px.bar(df, x='Normalized_Score', y='Feature', orientation='h',
                 title=f'Feature Selection — {method_name}',
                 color='Normalized_Score',
                 color_continuous_scale=['#e0e0e0', '#1DB954'],
                 text=df['Score'].round(4))
    fig.update_traces(textposition='outside', textfont_size=11)
    fig.update_layout(coloraxis_showscale=False,
                      xaxis_title='Normalized Score (0–1)',
                      height=max(350, len(df) * 28), **_WHITE)
    fig.update_xaxes(**_AXIS, range=[0, 1.15])
    fig.update_yaxes(**_AXIS)
    return fig


def plot_fs_comparison(results: dict):
    """Grouped bar chart comparing Normalized_Score across multiple FS methods.

    Parameters
    ----------
    results : dict — {method_label: imp_df} from run_feature_selection().
                     Automatically selects the top 15 features by max score.
    """
    all_feats = set(f for df in results.values() for f in df['Feature'])
    feat_max  = {}
    for feat in all_feats:
        feat_max[feat] = max(
            (df.loc[df['Feature'] == feat, 'Normalized_Score'].iloc[0]
             for df in results.values() if (df['Feature'] == feat).any()),
            default=0.0
        )
    top15   = sorted(feat_max, key=feat_max.get, reverse=True)[:15]
    palette = ['#1DB954', '#535353', '#22C55E', '#B3B3B3', '#16A34A', '#777']
    fig     = go.Figure()
    for i, (name, df) in enumerate(results.items()):
        scores = []
        for feat in top15:
            row = df[df['Feature'] == feat]
            scores.append(row['Normalized_Score'].iloc[0] if not row.empty else 0.0)
        fig.add_trace(go.Bar(name=name, x=top15, y=scores,
                              marker_color=palette[i % len(palette)],
                              text=[f'{s:.3f}' for s in scores],
                              textposition='outside',
                              textfont=dict(size=10, color='#111111')))
    fig.update_layout(barmode='group',
                       title='Feature Selection Comparison — Normalized Scores (Top 15)',
                       height=520,
                       legend=dict(orientation='h', yanchor='bottom', y=1.02,
                                   xanchor='right', x=1),
                       **_WHITE)
    fig.update_xaxes(**_AXIS, tickangle=35)
    fig.update_yaxes(**_AXIS, title='Normalized Score (0–1)', range=[0, 1.2])
    return fig


def plot_regression_performance(preds_df: pd.DataFrame, model_name: str):
    """Observed vs Predicted scatter for regression (Train and Test overlaid).

    Parameters
    ----------
    preds_df   : pd.DataFrame — Must contain columns: Observed, Predicted, Dataset.
    model_name : str          — Chart title prefix.
    """
    fig = px.scatter(preds_df, x='Observed', y='Predicted', color='Dataset',
                     color_discrete_map={'Train': '#535353', 'Test': '#1DB954'},
                     title=f'{model_name} — Observed vs Predicted', opacity=0.7)
    mn = min(preds_df['Observed'].min(), preds_df['Predicted'].min())
    mx = max(preds_df['Observed'].max(), preds_df['Predicted'].max())
    fig.add_shape(type='line', x0=mn, y0=mn, x1=mx, y1=mx,
                  line=dict(color='black', dash='dash'))
    return _apply_white(fig)


def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix'):
    """Plotly heatmap of the confusion matrix.

    Parameters
    ----------
    y_true  : array-like — True labels.
    y_pred  : array-like — Predicted labels.
    classes : list       — Sorted list of class names.
    title   : str        — Chart title.
    """
    cm  = confusion_matrix(y_true, y_pred)
    fig = px.imshow(cm, text_auto=True, x=classes, y=classes,
                    color_continuous_scale='Blues', title=title)
    fig.update_layout(xaxis_title='Predicted', yaxis_title='Actual', **_WHITE)
    return fig


def plot_metrics_summary(res_df: pd.DataFrame, task_type: str):
    """2×2 grouped bar chart — Train vs Test for every metric.

    Parameters
    ----------
    res_df    : pd.DataFrame — Output of run_training_pipeline() results.
    task_type : str          — 'Regression' or 'Classification'.

    Returns
    -------
    Plotly Figure with 4 subplots (one per metric), or None if no metrics found.
    """
    if task_type == 'Regression':
        candidates = [('R2', 'R²'), ('RMSE', 'RMSE'), ('MAE', 'MAE'),
                      ('Willmott_d', 'Willmott d')]
    else:
        candidates = [('Accuracy', 'Accuracy'), ('Precision', 'Precision'),
                      ('Recall', 'Recall'), ('F1', 'F1')]

    metrics = [(b, l) for b, l in candidates
               if f'Train_{b}' in res_df.columns and f'Test_{b}' in res_df.columns]
    if not metrics:
        return None

    n = len(metrics)
    rows, cols = (1, n) if n <= 2 else (2, 2)
    fig = make_subplots(rows=rows, cols=cols,
                         subplot_titles=[l for _, l in metrics],
                         vertical_spacing=0.22, horizontal_spacing=0.12)

    for idx, (base, lbl) in enumerate(metrics):
        r, c = divmod(idx, cols)
        shown = idx == 0
        for split, color in [('Train', '#535353'), ('Test', '#1DB954')]:
            fig.add_trace(go.Bar(
                x=res_df['Model'], y=res_df[f'{split}_{base}'].round(4),
                name=split, marker_color=color,
                legendgroup=split, showlegend=shown,
                text=res_df[f'{split}_{base}'].round(3),
                textposition='outside',
                textfont=dict(size=10, color='#111111')
            ), row=r + 1, col=c + 1)

    fig.update_layout(barmode='group',
                       title_text='Model Performance Summary — Train vs Test',
                       height=750,
                       legend=dict(orientation='h', yanchor='bottom', y=1.04,
                                   xanchor='right', x=1),
                       **_WHITE)
    fig.update_xaxes(**_AXIS, tickangle=30)
    fig.update_yaxes(**_AXIS)
    for ann in fig.layout.annotations:
        ann.font = dict(size=15, color='#111111')
    return fig


print('✓ Visualization functions defined')

---
<a id='3-dataset'></a>
## 📦 3. Dataset — CASP: Protein Structure Prediction

This demo uses the **CASP (Critical Assessment of protein Structure Prediction)** dataset, a well-known benchmark in bioinformatics.

| Property | Value |
|----------|-------|
| **Rows** | 45,730 |
| **Features** | F1 – F9 (9 physicochemical descriptors) |
| **Target** | RMSD — Root Mean Square Deviation of the predicted vs experimental protein structure |
| **Task** | Regression |

> **Dataset source:** [UCI ML Repository — CASP](https://archive.ics.uci.edu/ml/datasets/Physicochemical+Properties+of+Protein+Tertiary+Structure)  
> **Local path:** adjust `DATASET_PATH` below to match your environment.

In [ ]:
from pathlib import Path

# ── Adjust this path if needed ────────────────────────────────────────────────
DATASET_PATH = Path('../archive/protein.csv')

if not DATASET_PATH.exists():
    # Fallback: try common alternative locations
    for candidate in [
        Path('archive/protein.csv'),
        Path('./protein.csv'),
    ]:
        if candidate.exists():
            DATASET_PATH = candidate
            break
    else:
        raise FileNotFoundError(
            f'Dataset not found. Place protein.csv at: {DATASET_PATH.resolve()}'
        )

df = pd.read_csv(DATASET_PATH)
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

In [ ]:
eda = generate_eda(df)

print('=' * 60)
print(f'Shape  : {eda["shape"][0]:,} rows × {eda["shape"][1]} columns')
print(f'Numeric: {eda["numeric_cols"]}')
print(f'Categorical: {eda["categorical_cols"] or "none"}')
print()
print('Missing values:')
if len(eda['missing_values']) == 0:
    print('  None ✓')
else:
    print(eda['missing_values'])
print()
print('Descriptive statistics:')
eda['describe']

In [ ]:
# ── Correlation matrix ────────────────────────────────────────────────────────
# Strong positive correlation between F1/F2/F5 is expected — these features
# share physicochemical information. RMSD has moderate correlation with F4.
fig_corr = plot_correlation_matrix(df)
fig_corr.show()

In [ ]:
# ── Distribution and scatter ──────────────────────────────────────────────────
fig_hist = plot_histogram(df, col='RMSD')
fig_hist.show()

fig_scat = plot_scatter(df, x_col='F1', y_col='RMSD')
fig_scat.show()

---
<a id='4-feature-selection'></a>
## 🎯 4. Feature Selection Analysis

We run all 6 methods on the same dataset and compare which features consistently rank as most informative. **Consensus across methods** is a strong signal that the feature is genuinely predictive rather than an artifact of one particular algorithm.

Target variable: `RMSD`  
Features: all 9 numeric columns except the target.

In [ ]:
TARGET     = 'RMSD'
NUM_COLS   = [c for c in eda['numeric_cols'] if c != TARGET]
X_fs       = df[NUM_COLS]
y_fs       = df[TARGET]
TASK_TYPE  = 'Regression'

fs_results = {}
for label, key in tqdm(FEATURE_SELECTION_METHODS.items(), desc='Feature selection'):
    try:
        fs_results[label] = run_feature_selection(X_fs, y_fs, TASK_TYPE, key)
        print(f'  ✓ {label}')
    except Exception as e:
        print(f'  ✗ {label}: {e}')

print(f'\nCompleted {len(fs_results)}/{len(FEATURE_SELECTION_METHODS)} methods')

In [ ]:
# ── Individual method charts ──────────────────────────────────────────────────
for label, result_df in fs_results.items():
    fig = plot_fs_single(result_df, label)
    fig.show()

# ── Cross-method comparison ───────────────────────────────────────────────────
if len(fs_results) >= 2:
    print('\n--- Cross-method comparison ---')
    fig_cmp = plot_fs_comparison(fs_results)
    fig_cmp.show()

In [ ]:
# ── Select features via cumulative importance (95% threshold) ─────────────────
# Using ExtraTrees as the primary reference method
ref_method  = 'ExtraTrees Importance'
ref_df      = fs_results.get(ref_method, next(iter(fs_results.values())))

SELECTED_FEATURES = ref_df[ref_df['Cumulative_Importance'] <= 0.95]['Feature'].tolist()
if not SELECTED_FEATURES:
    SELECTED_FEATURES = ref_df['Feature'].head(5).tolist()

print(f'Method used for selection: {ref_method}')
print(f'Selected features ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}')
print()
print(ref_df[['Feature', 'Normalized_Score', 'Rank', 'Cumulative_Importance']].to_string(index=False))

---
<a id='5-training'></a>
## 🏋️ 5. Model Training & Hyperparameter Optimization

We demonstrate two HPO strategies side by side:

| Strategy | Settings |
|----------|----------|
| **Random Search** | `n_iter=25`, `cv=5` — tests 25 random parameter combinations |
| **Optuna TPE** | `n_trials=15` (demo; use 50–100 for production) — learns from each trial |

To keep runtime reasonable, training uses a **3-model subset**: Random Forest, Gradient Boosting, and XGBoost. Remove `selected_models` to train all available models.

In [ ]:
DEMO_MODELS = ['Random Forest', 'Gradient Boosting']
if _HAS_XGB:
    DEMO_MODELS.append('XGBoost')

print('Training with Random Search HPO...')
results_rs, preds_rs, models_rs, _ = run_training_pipeline(
    df              = df,
    features_cols   = SELECTED_FEATURES,
    target_col      = TARGET,
    task_type       = TASK_TYPE,
    scaler_option   = 'StandardScaler',
    selected_models = DEMO_MODELS,
    hpo_method      = 'random_search',
)

print('\n--- Random Search Results ---')
display_cols = ['Model'] + [c for c in results_rs.columns if c.startswith('Test_')]
print(results_rs[display_cols].to_string(index=False))

In [ ]:
if _HAS_OPTUNA:
    print('Training with Optuna TPE (15 trials/model)...')
    results_opt, preds_opt, models_opt, studies_opt = run_training_pipeline(
        df              = df,
        features_cols   = SELECTED_FEATURES,
        target_col      = TARGET,
        task_type       = TASK_TYPE,
        scaler_option   = 'StandardScaler',
        selected_models = DEMO_MODELS,
        hpo_method      = 'optuna',
        n_trials        = 15,
    )
    print('\n--- Optuna TPE Results ---')
    display_cols = ['Model'] + [c for c in results_opt.columns if c.startswith('Test_')]
    print(results_opt[display_cols].to_string(index=False))

    print('\n--- Best Hyperparameters Found ---')
    if studies_opt:
        for mname, study in studies_opt.items():
            print(f'\n  {mname} (best CV R²={study.best_value:.4f}):')
            for k, v in study.best_params.items():
                print(f'    {k}: {v}')
else:
    print('Optuna not installed. Install with: pip install optuna')
    print('Using Random Search results for the rest of the notebook.')
    results_opt, preds_opt, models_opt, studies_opt = results_rs, preds_rs, models_rs, None

# Use whichever results are available for the subsequent analysis
results_df = results_opt if _HAS_OPTUNA else results_rs
preds_df   = preds_opt   if _HAS_OPTUNA else preds_rs

---
<a id='6-results'></a>
## 📈 6. Results & Performance Evaluation

We evaluate each model across four complementary metrics:

- **R²** — overall explanatory power
- **RMSE** — penalises large errors (same units as target)
- **MAE** — robust average absolute error
- **Willmott d** — normalised agreement index (less sensitive to outliers than RMSE)

A model is considered reliable when **Train and Test metrics are close** — large gaps indicate overfitting.

In [ ]:
print('=' * 70)
print('MODEL LEADERBOARD (sorted by Test R²)')
print('=' * 70)
print(results_df.round(4).to_string(index=False))

# ── Metrics summary chart ─────────────────────────────────────────────────────
fig_summary = plot_metrics_summary(results_df, task_type=TASK_TYPE)
if fig_summary:
    fig_summary.show()

In [ ]:
# ── Observed vs Predicted — top 3 models ─────────────────────────────────────
for model_name in results_df['Model'].head(3).tolist():
    model_preds = preds_df[preds_df['Model'] == model_name]
    if TASK_TYPE == 'Regression':
        fig = plot_regression_performance(model_preds, model_name)
    else:
        test_preds = model_preds[model_preds['Dataset'] == 'Test']
        classes    = sorted(test_preds['Observed'].unique())
        fig = plot_confusion_matrix(test_preds['Observed'],
                                     test_preds['Predicted'],
                                     classes, f'Confusion Matrix: {model_name}')
    fig.show()

In [ ]:
# ── Optuna optimization history ───────────────────────────────────────────────
if studies_opt:
    for mname, study in studies_opt.items():
        fig = optuna.visualization.plot_optimization_history(study)
        fig.update_layout(title=f'Optuna — {mname} Optimization History',
                           paper_bgcolor='white', plot_bgcolor='white')
        fig.show()
        print(f'{mname}: {len(study.trials)} trials | best CV = {study.best_value:.4f}')

---
<a id='7-export'></a>
## 💾 7. Export & Deployment

MPP supports two export formats:

| Format | Use case |
|--------|----------|
| **Excel (.xlsx)** | Share metrics + predictions with collaborators or include in papers |
| **Joblib (.joblib)** | Serialise a fitted model for deployment or reuse in future experiments |

In [ ]:
import joblib

# ── Excel export ──────────────────────────────────────────────────────────────
def export_to_excel(results_df: pd.DataFrame, predictions_df: pd.DataFrame,
                    output_path: str = 'ML_results.xlsx') -> None:
    """Save results and predictions to an Excel workbook.

    Parameters
    ----------
    results_df     : pd.DataFrame — Per-model metrics (from run_training_pipeline).
    predictions_df : pd.DataFrame — Observed vs Predicted (from run_training_pipeline).
    output_path    : str          — Destination file path (default: ML_results.xlsx).
    """
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        results_df.to_excel(writer,     sheet_name='Metrics',     index=False)
        predictions_df.to_excel(writer, sheet_name='Predictions', index=False)
    print(f'Exported to: {Path(output_path).resolve()}')


export_to_excel(results_df, preds_df, output_path='ML_results.xlsx')

# ── Model serialization ───────────────────────────────────────────────────────
best_model_name  = results_df['Model'].iloc[0]
best_fitted      = (models_opt if _HAS_OPTUNA else models_rs).get(best_model_name)

if best_fitted:
    save_path = f'{best_model_name.replace(" ", "_")}.joblib'
    joblib.dump(best_fitted, save_path)
    print(f'Best model saved: {save_path}')

    # ── Reload and verify ─────────────────────────────────────────────────────
    loaded_model = joblib.load(save_path)
    X_test_sample = df[SELECTED_FEATURES].head(5)
    y_true_sample = df[TARGET].head(5).values
    y_pred_sample = loaded_model.predict(scale_features(X_test_sample, 'StandardScaler')[0])
    print('\nVerification — first 5 rows:')
    for t, p in zip(y_true_sample, y_pred_sample):
        print(f'  True: {t:.4f}  |  Predicted: {p:.4f}')

---
## 📄 Citation

If you use MPP in your research, please cite:

```
Barboza, T.O.C. (2025). ML Package Platform (MPP): A No-Code Machine Learning
Framework for Precision Agriculture. GitHub: https://github.com/thiagoagro
```

### References

- Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.
- Chen & Guestrin (2016). XGBoost: A Scalable Tree Boosting System. *KDD 2016*.
- Ke et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. *NeurIPS*.
- Akiba et al. (2019). Optuna: A Next-generation Hyperparameter Optimization Framework. *KDD 2019*.
- Willmott, C.J. (1981). On the validation of models. *Physical Geography*, 2(2), 184–194.

---
*Generated with ML Package Platform — Thiago Orlando Costa Barboza · ORCID: 0000-0001-5156-2474*